# XGBoost Hyperparameter Tuning - Manual Random Search

In [1]:
import numpy as np
import pandas as pd
from xgboost import XGBClassifier
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score

In [2]:
X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print(X_train.shape, X_test.shape)

(455, 30) (114, 30)


In [3]:
rng = np.random.default_rng(42)

space = {
    'n_estimators':     lambda: int(rng.integers(100, 500)),
    'max_depth':        lambda: int(rng.integers(3, 10)),
    'learning_rate':    lambda: float(10 ** rng.uniform(-2, -0.5)),  # log scale
    'subsample':        lambda: float(rng.uniform(0.6, 1.0)),
    'colsample_bytree': lambda: float(rng.uniform(0.6, 1.0)),
    'gamma':            lambda: float(rng.uniform(0, 5)),
    'min_child_weight': lambda: int(rng.integers(1, 8)),
    'reg_lambda':       lambda: float(rng.uniform(0.5, 5)),
}

In [4]:
def random_search(n_iter=20, cv=3):
    results = []
    for i in range(n_iter):
        params = {k: sample() for k, sample in space.items()}   # random combo
        model = XGBClassifier(**params, eval_metric='logloss',
                              random_state=42, n_jobs=-1)
        score = cross_val_score(model, X_train, y_train,
                                cv=cv, scoring='accuracy').mean()
        results.append({**params, 'cv_score': score})
        print(f'Iter {i+1}/{n_iter} -> CV accuracy: {score:.4f}')
    return pd.DataFrame(results).sort_values('cv_score', ascending=False).reset_index(drop=True)

In [5]:
results = random_search(n_iter=20, cv=3)
results.head()

Iter 1/20 -> CV accuracy: 0.9538
Iter 2/20 -> CV accuracy: 0.9494
Iter 3/20 -> CV accuracy: 0.9472
Iter 4/20 -> CV accuracy: 0.9494
Iter 5/20 -> CV accuracy: 0.9538
Iter 6/20 -> CV accuracy: 0.9670
Iter 7/20 -> CV accuracy: 0.9560
Iter 8/20 -> CV accuracy: 0.9648
Iter 9/20 -> CV accuracy: 0.9604
Iter 10/20 -> CV accuracy: 0.9472
Iter 11/20 -> CV accuracy: 0.9472
Iter 12/20 -> CV accuracy: 0.9450
Iter 13/20 -> CV accuracy: 0.9516
Iter 14/20 -> CV accuracy: 0.9604
Iter 15/20 -> CV accuracy: 0.9626
Iter 16/20 -> CV accuracy: 0.9494
Iter 17/20 -> CV accuracy: 0.9605
Iter 18/20 -> CV accuracy: 0.9538
Iter 19/20 -> CV accuracy: 0.9538
Iter 20/20 -> CV accuracy: 0.9670


,n_estimators,max_depth,learning_rate,subsample,colsample_bytree,gamma,min_child_weight,reg_lambda,cv_score
0,487,9,0.050621,0.675789,0.651969,2.378525,3,1.521092,0.967033
1,133,8,0.055079,0.731944,0.657810,0.517015,1,3.144401,0.967018
2,432,8,0.027070,0.872998,0.655901,0.999541,3,0.533130,0.964825
3,298,7,0.046691,0.752408,0.720605,3.151413,5,0.894425,0.962632
4,205,8,0.118942,0.779745,0.708897,0.481955,4,2.550993,0.960454


In [6]:
best_params = results.iloc[0].drop('cv_score').to_dict()
for k in ['n_estimators', 'max_depth', 'min_child_weight']:
    best_params[k] = int(best_params[k])
print('Best params:', best_params)
print('Best CV score:', round(results.loc[0, 'cv_score'], 4))

Best params: {'n_estimators': 487, 'max_depth': 9, 'learning_rate': 0.0506213492908261, 'subsample': 0.6757885436337143, 'colsample_bytree': 0.6519686021341886, 'gamma': 2.3785246311296686, 'min_child_weight': 3, 'reg_lambda': 1.5210920707289786}
Best CV score: 0.967


In [7]:
base = XGBClassifier(eval_metric='logloss', random_state=42).fit(X_train, y_train)
tuned = XGBClassifier(**best_params, eval_metric='logloss', random_state=42).fit(X_train, y_train)

print('Default XGBoost test accuracy:', round(accuracy_score(y_test, base.predict(X_test)), 4))
print('Tuned XGBoost test accuracy  :', round(accuracy_score(y_test, tuned.predict(X_test)), 4))

Default XGBoost test accuracy: 0.9561
Tuned XGBoost test accuracy  : 0.9561
